# Mining Accident Investigation System — Kostenko Demonstration

**Test case:** Kostenko mine explosion, Kazakhstan, 28 October 2023.  
**Pipeline:** `v1 (load) → v2 (classify) → v3 (CBR)`  
**Status:** Deterministic backbone complete; LLM-backed v4–v6 are the thesis contribution still to be implemented.

This notebook walks through each subsystem cell-by-cell, showing what real data looks like at each pipeline stage. Run cells top to bottom.

## Setup

Add `src/` to `sys.path` and import the public surface of each subsystem. Same path setup that `tests/conftest.py` and `scripts/demo_kostenko.py` use.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import DEFAULT_REGULATORY_KB, DEFAULT_KOSTENKO_KB
from kb.store import KnowledgeBase
from v1_decomposition import decompose_from_json
from v2_identification import classify
from v3_precedent_matching import match_precedents

print("Project root:", PROJECT_ROOT)
print("Regulatory KB:", DEFAULT_REGULATORY_KB.name)
print("Active case:  ", DEFAULT_KOSTENKO_KB.name)

## 1. Knowledge base

The KB has three layers (see `note.md` § "organized evidence room"):

1. **Domain knowledge** — case-agnostic regulations + cause taxonomy (Rostechnadzor)
2. **Precedent cases** — past accidents with `similarity_profile` for runtime CBR
3. **Active case** — the Kostenko investigation arguments + ground truth

All three are loaded into a single `KnowledgeBase` store with referential integrity checks.

In [ ]:
kb = KnowledgeBase.from_files(
    regulatory_path=DEFAULT_REGULATORY_KB,
    case_path=DEFAULT_KOSTENKO_KB,
    case_name="kostenko",
)
kb.summary()

## 2. v1 — Decomposition

**Input:** Kostenko JSON file with arguments already extracted into the 8-field Toulmin schema (`id`, `source`, `topic`, `claim`, `evidence`, `warrant`, `confidence`, `cause_categories`).  
**Output:** A `CaseFile` with `.arguments` (pipeline input for v2-v6) and `.ground_truth` (held aside for v5 evaluation).

Three independent expert investigations contributed arguments:

In [ ]:
case = decompose_from_json(DEFAULT_KOSTENKO_KB)

print(f"Total arguments: {len(case.arguments)}\n")
by_source = Counter(a.source for a in case.arguments)
for src, count in sorted(by_source.items()):
    src_meta = next((s for s in case.metadata.sources if s.get('id') == src), {})
    print(f"  {src}: {count:2d} args  -- {src_meta.get('full_name', '?')}")

print(f"\nGround truth (held aside for v5 evaluation):")
print(f"  {len(case.ground_truth.attack_relations):2d} attack relations")
print(f"  {len(case.ground_truth.support_relations):2d} support relations")
print(f"  {len(case.ground_truth.open_questions):2d} open questions")

In [ ]:
# Inspect one argument to see the full Toulmin schema in action
sample = case.arguments[0]
print(f"Argument {sample.id}")
print(f"  source:           {sample.source}")
print(f"  topic:            {sample.topic}")
print(f"  claim:            {sample.claim}")
print(f"  evidence:         {sample.evidence[:120]}...")
print(f"  warrant:          {sample.warrant[:120]}...")
print(f"  confidence:       {sample.confidence}")
print(f"  cause_categories: {sample.cause_categories}")

## 3. v2 — Identification (rule-based classification)

**Method:** No LLM. Aggregate `cause_categories` across all arguments, then map each cause to its associated accident types via the regulatory KB itself (no hand-coded table — `regulation.relevant_cause_categories × regulation.applies_to_accident_types`). Most-voted accident type = primary; runners-up at ≥ 50% of top votes = secondary.

**Why not an LLM?** Classification isn't the thesis contribution — Markarian's framework already handles it. v2 is intentionally simple.

In [ ]:
classif = classify(case.arguments, kb.regulations)

print(f"Primary type:    {classif.primary_type}")
print(f"Secondary types: {classif.secondary_types}")

In [ ]:
# Cause profile: how often each cause category appears across the 21 arguments
print("Cause profile:")
for cid, count in sorted(classif.cause_profile.items(), key=lambda x: (-x[1], x[0])):
    cat = kb.get_cause_category(cid)
    label = cat.label if cat else "?"
    bar = "#" * count
    print(f"  {cid}  {label:<32}  {bar} ({count})")

In [ ]:
# Type votes: how each cause-category occurrence translates into accident type votes
print("Type votes (full ranking):")
for type_label, votes in sorted(classif.type_votes.items(), key=lambda x: (-x[1], x[0])):
    bar = "#" * max(votes // 2, 1)
    print(f"  {type_label:<25}  {bar} ({votes})")

## 4. v3 — Precedent matching (two-step CBR)

**Method:**
1. **Type filter** — keep precedents whose `accident_type` matches v2's primary or secondary types
2. **Jaccard scoring** — `|shared cause_categories| / |union cause_categories|` between the case and each surviving precedent

Result is ranked by overlap, with funnel telemetry (how many precedents survive each stage).

In [ ]:
match_result = match_precedents(classif, kb.precedents)

print(f"Funnel: {match_result.total_precedents} precedents in KB"
      f" -> {match_result.filtered_count} passed type filter\n")

for i, m in enumerate(match_result.matches, 1):
    prec = next(p for p in kb.precedents if p.id == m.precedent_id)
    print(f"{i}. {m.precedent_id} — {prec.mine}")
    print(f"   accident_type:  {m.accident_type} (matched_via: {m.matched_via})")
    print(f"   overlap_score:  {m.overlap_score:.4f}")
    print(f"   shared causes:  {m.shared_cause_categories}")
    print(f"   fatalities:     {prec.fatalities}")
    print()

### Interpreting the result

**The ranking is correct on day one.** Listviazhnaya — the deadliest Russian mining disaster in years (51 fatalities, also a longwall methane explosion) — is the system's top match for Kostenko. Alardinskaya, also a methane fire in goaf, is second.

**Absolute Jaccard scores are low (0.09 and 0.08).** This is a transparent artifact of the current data: Kostenko's arguments carry only technical (`TC-*`) tags, while Listviazhnaya is heavily organizational (`OC-01`, `OC-04`). The intersection collapses to just `TC-01` (methane accumulation). Two paths improve scores without code changes:

1. Re-tag Kostenko arguments with organizational causes (a separate data task — would require re-reading the original PDFs)
2. Add more precedents tagged similarly to Kostenko (e.g. detailed UBB extraction)

**The thesis defense framing:** ranking correctness is independent of absolute score; scores will rise as the KB grows. The system is designed to scale this way.

## 5. Ground truth — what v5 will be evaluated against

The Kostenko KB ships with manually-annotated attack relations, support relations, and open questions extracted from the three expert reports. These are **never fed into v1–v5**; they sit aside as the evaluation target. v5 (Dung's argumentation framework) is considered correct if its computed conflict graph matches this annotation.

In [ ]:
print("ATTACK RELATIONS:")
for atk in case.ground_truth.attack_relations:
    print(f"  [{atk.id}] {atk.attacker} --({atk.type})--> {atk.target}")
    print(f"    {atk.description[:140]}...\n")

print("SUPPORT RELATIONS:")
for sup in case.ground_truth.support_relations:
    print(f"  [{sup.id}] {sup.topic}")
    print(f"    supporters: {sup.supporters}  ({sup.strength})\n")

print("OPEN QUESTIONS:")
for oq in case.ground_truth.open_questions:
    print(f"  [{oq.id}] {oq.question}")
    print(f"    raised by: {oq.raised_by}\n")

## What's next — the thesis contribution

Everything above is the deterministic scaffold (90 passing tests). The thesis weight lives in the next three subsystems:

- **v4 — 4 specialist LLM agents** (Technical, Organizational, Challenger, Regulatory) with intentionally overlapping scope on causal claims, producing arguments in the same 8-field schema as v1.
- **v5 — Dung's argumentation framework** built over the combined claim set from v1 + v4. Topic-based filter then LLM-confirmed conflict pairs become attacks. NetworkX computes grounded and preferred extensions; the comparison itself is a finding (consensus vs genuine ambiguity).
- **v6 — explainable report generation.** LLM populates a 7-section report from v5's structured output, with full traceability from each conclusion back to evidence → agent → attack structure.

Run `pytest` from the project root to verify the deterministic scaffold below. 90 tests should pass in well under a second.